# 01 · The data — questions whose answer is a table cell

**The thesis: some answers exist only inside a table, and that makes them a
different kind of question.** This notebook loads the real benchmark we use,
shows one question end to end, and introduces the single split the whole paper
is reported on: *lookup* versus *compute*.

No API key needed. Nothing here calls a model.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `load_domain(d)` | read cached (question, gold, table) triples for a domain | `load_domain('Biology')` → 460 triples |
| `Triple` | one benchmark item: question, gold answer, table, labels | `t.question`, `t.gold`, `t.table_csv` |
| `task_family(t)` | coarse split used in every result | `Lookup` or `Compute` |
| `is_grounded(g,tbl)` | is the gold answer actually present in its table? | `True` / `False` |

## Step 1 — make the local modules importable

The kernel starts in this notebook's own directory, which is also where the
project's modules live, so an empty path entry is enough. We print the dataset
cache location to confirm we are reading from disk and not the network.

In [1]:
import sys, os
sys.path.insert(0, "")

import scitableqa_loader as L
print("cache dir :", L.CACHE)
print("exists    :", os.path.isdir(L.CACHE))

cache dir : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-foundations-and-methods/anacodic-agentic-cookbook/02-experiments/rashan-table-agent/data/scitableqa
exists    : True


## Step 2 — load one domain and look at the counts

`load_domain` reads the **local cache only**. It raises if a domain was never
downloaded, rather than silently fetching — an evaluation should not depend on
the network mid-run.

In [2]:
trips = L.load_domain("Biology")
print("triples:", len(trips))
print("fields :", list(vars(trips[0]).keys()))

[warn] 15 QA items had no cached table (skipped)
triples: 460
fields : ['question', 'gold', 'reference_answer', 'reasoning_type', 'table_csv', 'table_path', 'domain', 'task_family', 'grounded']


## Step 3 — one real item, end to end

This is the whole task in one screen: a question, the table it refers to, and
the answer a human wrote down. Note that the answer is a **cell** — the article
text around this table never states it.

In [3]:
from table_tools import csv_to_markdown

t = next(x for x in trips if x.task_family == "Lookup" and x.grounded)
print("QUESTION :", t.question)
print("GOLD     :", t.gold)
print("TYPE     :", t.reasoning_type, "|", t.task_family)
print()
print(csv_to_markdown(t.table_csv)[:700])

QUESTION : What is the forward primer sequence for Exon 3 & Exon 4?
GOLD     : ACGTAGTGCATACACCCTTG
TYPE     : Cell Selection | Lookup

| UGT1A1 region | Forward primer (5' 3') | Reverse primer (5' 3') |
| --- | --- | --- |
| Promoter & Exon 1 | GAAACCTAATAAAGCTCCACCTTC | TTGCTCAGCATATATCTGGGGC |
| Exon 2 | TCATTTAAAGGGACCACGCC | GGAAAAGCCAAATCTAAGGTTCC |
| Exon 3 & Exon 4 | ACGTAGTGCATACACCCTTG | GAAACAACGCTATTAAATGCTACG |
| Exon 5 | GAAACAGGTTTCCTTTCCCAAG | CAGAGGGGGCACGATACATA |


## Step 4 — the split everything is reported on

Two kinds of question, and they are genuinely different problems:

```
LOOKUP    "How many donors were in the unknown group?"   -> FIND the cell
COMPUTE   "What percentage of the total is that?"        -> find cells, CALCULATE
```

We never report a single blended accuracy, because a system can be excellent at
one and hopeless at the other.

In [4]:
from collections import Counter
fam = Counter(x.task_family for x in trips)
print("by family:", dict(fam))
print()
for rt, n in Counter(x.reasoning_type for x in trips).most_common(8):
    print(f"  {rt:28s} {n}")

by family: {'Compute': 264, 'Lookup': 196}

  Cell Selection               175
  Arithmetic                   135
  Calculation                  77
  Counting                     19
  Percentage Calculation       13
  Identification               5
  Arithmetic Calculation       5
  Correlation Calculation      5


## Step 5 — a compute question, for contrast

Here the answer must be derived. Notice it is *not* a cell you can point at.

In [5]:
tc = next(x for x in trips if x.task_family == "Compute" and x.grounded)
print("QUESTION :", tc.question)
print("GOLD     :", tc.gold)
print()
print(csv_to_markdown(tc.table_csv)[:520])

QUESTION : What percentage of oocytes tested in the category 'Female carriers of structural rearrangements or monogenic disorders' had premeiotic aneuploidy?
GOLD     : 6.66%

| Classification of female partners based on reproductive histories | Total No. of donors | Total oocytes tested | Oocytes with premeiotic aneuploidy | No. of donors contributing oocytes with premeiotic aneuploidy | No. of oocytes with Simple (SE)/ Complex (CE) Errors |
| --- | --- | --- | --- | --- | --- |
| 4.Oocyte preservation due to breast cancer | 2 | 10 | 5 (50%) | 2 donors | 1(SE); 4(CE) |
| 5.Female carriers of structural rearrangements or monogenic disorders (non- cancer related) | 17 | 30 | 2 (6.66%) | 2


## Step 6 — the label check, and a lesson about your own measurements

A *lookup* answer must by definition appear in its own table. We test that and
call it `grounded`. It bounds the accuracy any system can reach, so without it
you can mistake **bad labels for a bad model**.

But read the next cell carefully, because this check was itself wrong the first
time. A strict substring test says ~80% of lookup golds are present. A gold of
`3.5 years` against a cell containing `3.5` fails that test even though the
value *is* there. Comparing on the numeric/cell level instead gives ~98%.

**The benchmark is largely clean. Our first measurement was the bug.** We nearly
built a paper around the 80%.

In [6]:
import re, csv, io
from table_tools import _clean

def cells(tbl):
    out=[]
    for r in csv.reader(io.StringIO(tbl)):
        out += [_clean(v) for v in r if _clean(v)]
    return out
def nrm(s): return re.sub(r"[^a-z0-9.]", "", str(s).lower())
def nums(s): return set(re.findall(r"-?\d+(?:\.\d+)?", str(s).replace(",", "")))

lk = [x for x in trips if x.task_family == "Lookup"]
strict = sum(1 for x in lk if x.grounded)          # the loader's conservative flag

lenient = 0
for x in lk:
    cs = cells(x.table_csv); cn = [nrm(c) for c in cs]
    g = nrm(x.gold); gn = nums(x.gold)
    hit = any(g and (g == c or g in c or c in g) for c in cn)
    if not hit and gn:
        tabnums = set()
        for c in cs: tabnums |= nums(c)
        hit = gn.issubset(tabnums)
    lenient += hit

print(f"LOOKUP items: {len(lk)}")
print(f"  strict substring test : {strict}/{len(lk)} = {strict/len(lk):.0%} grounded")
print(f"  cell / number match   : {lenient}/{len(lk)} = {lenient/len(lk):.0%} grounded")
print()
print("The gap is unit words: gold '3.5 years' vs a cell holding '3.5'.")
print("Lesson: validate the validator before you report a number from it.")

LOOKUP items: 196
  strict substring test : 157/196 = 80% grounded
  cell / number match   : 192/196 = 98% grounded

The gap is unit words: gold '3.5 years' vs a cell holding '3.5'.
Lesson: validate the validator before you report a number from it.


## Step 7 — a case where the gold cannot be reached

We found items whose gold answer is not obtainable from the table as shipped.
The cleanest example: the gold says 48, but the two sequences in the table are
24 and 22 characters, so the table supports 46.

This is why a human audit is part of the project, and why we report a verifiable
subset alongside the full set.

In [7]:
prim = [x for x in trips if "concatenating the forward and reverse" in x.question]
if prim:
    p = prim[0]
    import csv, io
    from table_tools import _clean
    rows = [[_clean(v) for v in r] for r in csv.reader(io.StringIO(p.table_csv))]
    rows = [r for r in rows if any(r)]
    fwd, rev = rows[1][1], rows[1][2]
    print("question :", p.question[:88], "...")
    print("gold     :", p.gold)
    print(f"forward  : {len(fwd)} chars")
    print(f"reverse  : {len(rev)} chars")
    print(f"table supports: {len(fwd)} + {len(rev)} = {len(fwd)+len(rev)}")
else:
    print("(sample not in this slice)")

question : What is the total length of the sequence formed by concatenating the forward and reverse ...
gold     : 48 nucleotides
forward  : 24 chars
reverse  : 22 chars
table supports: 24 + 22 = 46


## What to carry out of this notebook

```
 · the unit of work is a TRIPLE: question + gold answer + table
 · every result is split LOOKUP vs COMPUTE, never blended
 · `grounded` is a CONSERVATIVE flag; the benchmark is ~98% clean on lookup
 · a handful of items genuinely cannot be answered from the table as shipped
 · validate your validator before you report a number from it
```

Next: `02_scoring.ipynb` — how we decide whether an answer is correct.